In [5]:
import numpy as np
import pandas as pd
import argparse
import numpy as np
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from ct_rep.utils.sklearn_classifiers import knn_on_embeddings
from ct_rep.utils.pytorch_mlp import Classifier
import os
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from ct_rep.celltype.celltype_pred import evaluate
from sklearn.decomposition import PCA


In [6]:
from ct_rep.utils.utils import extract_baseline_embs, split_train_test
from ct_rep.utils.utils import sample_data, adapt_vars
from ct_rep.utils.utils import normalize, split_train_test

In [7]:
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 10

In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
results_path = "ct_rep/celltype/within_dataset_results.csv"

results_df = pd.read_csv(results_path)
columns=['config', 'tissue', 'dataset_ref', 'filename_ref', 'dataset_query', 'filename_query', 'classifier', 'method', 'emb_path', 'accuracy', 'f1_score_weighted', 'f1_score_macro']
assert results_df.columns.tolist() == columns
print(results_df)
# results_df = pd.DataFrame(columns = columns)

                                              config tissue  \
0  [atlas_train]_[abc_atlas_train.h5ad]_[None]_[N...  brain   

            dataset_ref          filename_ref       dataset_query  \
0  abc_atlas_train.h5ad  abc_atlas_train.h5ad  abc_atlas_val.h5ad   

       filename_query classifier     method                       emb_path  \
0  abc_atlas_val.h5ad        knn  scConcept  small_weighted__hgld4ax1_last   

   accuracy  f1_score_weighted  f1_score_macro  
0  0.925233           0.921935        0.753647  


In [6]:
from ct_rep.celltype.celltype_pred import run

dataset_configs = [
    ("atlas_train", "abc_atlas_train.h5ad", None, None, "class", "atlas", "abc_atlas_val.h5ad", None, None, "class", "brain"),
]
results = []

for dataset_config in dataset_configs:
    config = "_".join([f'[{c}]' for c in dataset_config])
    dataname_ref, filename_ref, split_key_ref, split_value_ref, label_key_ref, dataname_query, filename_query, split_key_query, split_value_query, label_key_query, tissue = dataset_config
    print(f"config: {dataset_config}")

    adata_path_ref = f"/p/project1/hai_fzj_bda/spitzer2/point_transformer/data/raw/concept_embeddings/train_val_data/{filename_ref}"
    adata_path_query = f"/p/project1/hai_fzj_bda/spitzer2/point_transformer/data/raw/concept_embeddings/train_val_data/{filename_query}"
    
    emb_paths = [
        #("scConcept", "cosine", 0.0005, "small_weighted__hgld4ax1_last"),
        #("scConcept", "cosine", 0.0005, "big_weighted__6iz4xrqm_last"),
        ("scConcept", "cosine", 0.0005, "big_weighted__uv28i4o3_last"),
    ]
    classifiers = ["knn"]
    min_count = 50
    train_size = 200000
    test_size = 60000

    for classifier in classifiers:
        for method, metric, lr, model_id in emb_paths:
            adata_train = sc.read_h5ad(adata_path_ref)
            adata_test = sc.read_h5ad(adata_path_query)
            
            if model_id != "":
                print(f"Embedding: {model_id}, lr={lr}, metric={metric}")
                emb_path_ref = f"/p/home/jusers/dipippo1/jureca/projects/scConcept-1/src/concept/concept_embeddings/models/{model_id}/{dataname_ref}/cell_embs_cls.npy"
                emb_path_query = f"/p/home/jusers/dipippo1/jureca/projects/scConcept-1/src/concept/concept_embeddings/models/{model_id}/{dataname_query}/cell_embs_cls.npy"

                if not os.path.exists(emb_path_ref) or not os.path.exists(emb_path_query):
                    print(f"File not found!!!!, skipping")
                    print(emb_path_ref)
                    continue

                adata_train.obsm['X_emb'] = np.load(emb_path_ref)
                adata_test.obsm['X_emb'] = np.load(emb_path_query)
            
            # remove low frequency cell types
            freq = adata_train.obs[label_key_ref].value_counts(normalize=False)
            print(f'removing {list(freq[freq < min_count].index)} from dataset')
            adata_train = adata_train[adata_train.obs[label_key_ref].isin(freq[freq > min_count].index)]
            adata_test = adata_test[adata_test.obs[label_key_query].isin(freq[freq > min_count].index)]
            
            adata_train = adata_train.copy()
            adata_test = adata_test.copy()

            # Subsample train e test
            if train_size is not None and adata_train.n_obs > train_size:
                sc.pp.subsample(adata_train, n_obs=train_size, random_state=42)
                print(f"Subsampled train to {train_size} cells")
            
            if test_size is not None and adata_test.n_obs > test_size:
                sc.pp.subsample(adata_test, n_obs=test_size, random_state=42)
                print(f"Subsampled test to {test_size} cells")

            if metric == 'cosine':
                adata_train.obsm['X_emb'] = F.normalize(torch.tensor(adata_train.obsm['X_emb']), p=2, dim=1).numpy()
                adata_test.obsm['X_emb'] = F.normalize(torch.tensor(adata_test.obsm['X_emb']), p=2, dim=1).numpy()
                
            train_embs = adata_train.obsm['X_emb']
            test_embs = adata_test.obsm['X_emb']
            train_labels = adata_train.obs[label_key_ref]
            test_labels = adata_test.obs[label_key_query]

            accuracy, f1_score_weighted, f1_score_macro, y_pred = evaluate(train_embs, train_labels, test_embs, test_labels, classifier, metric=metric, pred_within_test_labels=False, lr=lr)
            print(f'{accuracy:.3f}, {f1_score_weighted:.3f}, {f1_score_macro:.3f}')
            
            results.append((config, tissue, filename_ref, filename_ref, filename_query, filename_query, classifier, method, model_id, accuracy, f1_score_weighted, f1_score_macro))

config: ('atlas_train', 'abc_atlas_train.h5ad', None, None, 'class', 'atlas', 'abc_atlas_val.h5ad', None, None, 'class', 'brain')
Embedding: big_weighted__6iz4xrqm_last, lr=0.0005, metric=cosine
removing [] from dataset
Subsampled train to 200000 cells
Subsampled test to 60000 cells
train split: 200000 cells containing 35 cell types, embeddings of size 512
test split: 60000 cells containing 35 cell types, embeddings of size 512
Fitting KNN (cosine) with n_neighbors:  10
knn Accuracy: 0.962
knn F1-Weighted: 0.960
knn F1-Macro: 0.874
0.962, 0.960, 0.874


In [7]:

print(results_df)

duplicates = results_df.duplicated(subset=results_df.columns[:-3], keep='last')
print(f"Removing {duplicates.sum()} duplicates")
results_df = results_df[~duplicates]

                                              config tissue  \
0  [atlas_train]_[abc_atlas_train.h5ad]_[None]_[N...  brain   

            dataset_ref          filename_ref       dataset_query  \
0  abc_atlas_train.h5ad  abc_atlas_train.h5ad  abc_atlas_val.h5ad   

       filename_query classifier     method                       emb_path  \
0  abc_atlas_val.h5ad        knn  scConcept  small_weighted__hgld4ax1_last   

   accuracy  f1_score_weighted  f1_score_macro  
0  0.925233           0.921935        0.753647  
Removing 0 duplicates


In [8]:
results_path = "ct_rep/celltype/within_dataset_results.csv"
results_df.to_csv(results_path, index=False)

In [6]:
import scanpy as sc

adata = sc.read_h5ad("/p/project1/hai_fzj_bda/spitzer2/point_transformer/data/raw/Zhuang-ABCA-1.h5ad")
print(adata.obs.columns.tolist())

['abc_sample_id', 'brain_section_label', 'brain_section_label_adata', 'class', 'class_color', 'cluster', 'cluster_alias', 'cluster_color', 'cluster_confidence_score', 'donor_genotype', 'donor_label', 'donor_sex', 'feature_matrix_label', 'high_quality_transfer', 'neurotransmitter', 'neurotransmitter_color', 'parcellation_category', 'parcellation_category_color', 'parcellation_division', 'parcellation_division_color', 'parcellation_index', 'parcellation_organ', 'parcellation_organ_color', 'parcellation_structure', 'parcellation_structure_color', 'parcellation_substructure', 'parcellation_substructure_color', 'subclass', 'subclass_color', 'subclass_confidence_score', 'supertype', 'supertype_color', 'x', 'x_ccf', 'y', 'y_ccf', 'z', 'z_ccf']


In [7]:
import scanpy as sc

adata = sc.read_h5ad("/p/project1/hai_fzj_bda/spitzer2/point_transformer/data/raw/ISD-1.h5ad")
print(adata.obs.columns.tolist())

['fov', 'volume', 'center_x', 'center_y', 'min_x', 'min_y', 'max_x', 'max_y', 'anisotropy', 'transcript_count', 'perimeter_area_ratio', 'solidity', 'Fth1_raw', 'Fth1_high_pass', 'DAPI_raw', 'DAPI_high_pass', 'App_raw', 'App_high_pass', 'Aldoc_raw', 'Aldoc_high_pass', 'Sst_raw', 'Sst_high_pass', 'Plp1_raw', 'Plp1_high_pass', 'PolyT_raw', 'PolyT_high_pass', 'sample', 'broad_region', 'intermediate_region', 'detailed_region', 'cell_id']


In [ ]:
from ct_rep.celltype.celltype_pred import evaluate
from sklearn.model_selection import train_test_split

# Config: (dataname, filename, label_key, tissue)
dataset_configs = [
    ("zeng", "Zhuang-ABCA-1.h5ad", "class", "brain"),  # cambia con il tuo file
]
results = []

for dataset_config in dataset_configs:
    dataname, filename, label_key, tissue = dataset_config
    config = f"[{dataname}]_[{label_key}]"
    print(f"config: {dataset_config}")

    adata_path = f"/p/project1/hai_fzj_bda/spitzer2/point_transformer/data/raw/{filename}"
    
    emb_paths = [
        ("scConcept", "cosine", 0.0005, "big_weighted__uv28i4o3_last"),
    ]
    classifiers = ["knn"]
    min_count = 50
    train_size = 200000
    test_size = 60000
    test_split = 0.2  # 80% train, 20% test

    for classifier in classifiers:
        for method, metric, lr, model_id in emb_paths:
            adata = sc.read_h5ad(adata_path)
            
            if model_id != "":
                print(f"Embedding: {model_id}, lr={lr}, metric={metric}")
                emb_path = f"/p/home/jusers/dipippo1/jureca/projects/scConcept-1/src/concept/concept_embeddings/models/{model_id}/{dataname}/cell_embs_cls.npy"
                cell_ids_path = f"/p/home/jusers/dipippo1/jureca/projects/scConcept-1/src/concept/concept_embeddings/models/{model_id}/{dataname}/cell_ids.npy"
            
                if not os.path.exists(emb_path) or not os.path.exists(cell_ids_path):
                    print(f"File not found!!!!, skipping")
                    print(emb_path)
                    continue
            
                # Carica embeddings e cell_ids
                emb = np.load(emb_path)
                cell_ids = np.load(cell_ids_path, allow_pickle=True).astype(str)
                
                # Trova cellule comuni
                common_cells = adata.obs_names.intersection(cell_ids)
                print(f"Cells in adata: {adata.n_obs}, Cells with embeddings: {len(cell_ids)}, Common: {len(common_cells)}")
                
                if len(common_cells) == 0:
                    print("No common cells found, skipping")
                    continue
                
                # Filtra adata alle cellule comuni
                adata = adata[common_cells].copy()
                
                # Allinea embeddings
                emb_df = pd.DataFrame(emb, index=cell_ids)
                adata.obsm['X_emb'] = emb_df.loc[adata.obs_names].values
            
            # remove low frequency cell types
            freq = adata.obs[label_key].value_counts(normalize=False)
            print(f'removing {list(freq[freq < min_count].index)} from dataset')
            adata = adata[adata.obs[label_key].isin(freq[freq >= min_count].index)]
            adata = adata.copy()
            
            # Split train/test
            indices = np.arange(adata.n_obs)
            train_idx, test_idx = train_test_split(indices, test_size=test_split, random_state=42, stratify=adata.obs[label_key])
            
            adata_train = adata[train_idx].copy()
            adata_test = adata[test_idx].copy()
            
            # Subsample train e test
            if train_size is not None and adata_train.n_obs > train_size:
                sc.pp.subsample(adata_train, n_obs=train_size, random_state=42)
                print(f"Subsampled train to {train_size} cells")
            
            if test_size is not None and adata_test.n_obs > test_size:
                sc.pp.subsample(adata_test, n_obs=test_size, random_state=42)
                print(f"Subsampled test to {test_size} cells")

            if metric == 'cosine':
                adata_train.obsm['X_emb'] = F.normalize(torch.tensor(adata_train.obsm['X_emb']), p=2, dim=1).numpy()
                adata_test.obsm['X_emb'] = F.normalize(torch.tensor(adata_test.obsm['X_emb']), p=2, dim=1).numpy()
                
            train_embs = adata_train.obsm['X_emb']
            test_embs = adata_test.obsm['X_emb']
            train_labels = adata_train.obs[label_key]
            test_labels = adata_test.obs[label_key]

            print(f"Train: {len(train_labels)} cells | Test: {len(test_labels)} cells")

            accuracy, f1_score_weighted, f1_score_macro, y_pred = evaluate(train_embs, train_labels, test_embs, test_labels, classifier, metric=metric, pred_within_test_labels=False, lr=lr)
            print(f'{accuracy:.3f}, {f1_score_weighted:.3f}, {f1_score_macro:.3f}')
            
            results.append((config, tissue, dataname, filename, dataname, filename, classifier, method, model_id, accuracy, f1_score_weighted, f1_score_macro))

# Raw count / PCA

First select the dataset config and embedding from the previous section

In [31]:
# dataset_config = ("neurips_2021_multiome", "adata_gex.h5ad", "batch", "s1", "cell_type", "neurips_2021_multiome", "adata_gex.h5ad", "batch", "s3", "cell_type", "blood")
dataset_config = ("neurips_2021_multiome", "adata_gex.h5ad", "batch", "s1", "cell_type", "neurips_2021_multiome", "adata_gex_xenium_panel.h5ad", "batch", "s3", "cell_type", "blood")
# dataset_config = ("sea-ad", "adata_subsample.h5ad", "-", "*", "Subclass", "sea-ad", "adata_merfish_subsample.h5ad", "-", "*", "Subclass", "brain")
# dataset_config = ("10x_scRNA_ovarian_cancer", "adata.h5ad", "-", "*", "cell_type", "spatial_xenium_human_ovarian_cancer_prime_5k_panel_ffpe", "adata_subsample.h5ad", "-", "*", "cell_type", "artery")

config = "_".join([f'[{c}]' for c in dataset_config])
dataset_ref, filename_ref, split_key_ref, split_value_ref, label_key_ref, dataset_query, filename_query, split_key_query, split_value_query, label_key_query, tissue = dataset_config

adata_path_ref = f"/lustre/groups/ml01/projects/contrastive_transformer/data/{dataset_ref}/h5ads/{filename_ref}"
adata_path_query = f"/lustre/groups/ml01/projects/contrastive_transformer/data/{dataset_query}/h5ads/{filename_query}"

In [32]:
method = "Inner PCA-128 totalcount_log1p" # Inner PCA-128, Raw Count Inner
emb_path = "Inner PCA-128 totalcount_log1p" # Inner PCA-128, Raw Count Inner
normalization = "totalcount_log1p" # raw, log1p, totalcount_log1p
classifier = 'knn' # knn, linear
metric = 'euclidean'
lr = 0.0005
min_count = 50

In [ ]:
adata_train = sc.read_h5ad(adata_path_ref)
adata_test = sc.read_h5ad(adata_path_query)
# adata_train.obsm['X_emb'] = np.load(emb_path_ref)
# adata_test.obsm['X_emb'] = np.load(emb_path_query)

# remove low frequency cell types
freq = adata_train.obs[label_key_ref].value_counts(normalize=False)
print(f'removing {list(freq[freq < min_count].index)} from dataset')
adata_train = adata_train[adata_train.obs[label_key_ref].isin(freq[freq > min_count].index)]
adata_test = adata_test[adata_test.obs[label_key_query].isin(freq[freq > min_count].index)]

# split
adata_train, adata_test = split_train_test(adata_train, adata_test, split_key_ref, split_key_query, split_value_ref, split_value_query)

adata_train = adata_train.copy()
adata_test = adata_test.copy()

############################################################################################################

adata_train, adata_test = extract_baseline_embs(adata_train, adata_test, method, key='X_emb')

############################################################################################################
# if metric == 'cosine':
#     adata_train.obsm['X_emb'] = F.normalize(torch.tensor(adata_train.obsm['X_emb']), p=2, dim=1).numpy()
#     adata_test.obsm['X_emb'] = F.normalize(torch.tensor(adata_test.obsm['X_emb']), p=2, dim=1).numpy()

train_embs = adata_train.obsm['X_emb']
test_embs = adata_test.obsm['X_emb']

train_labels = adata_train.obs[label_key_ref]
test_labels = adata_test.obs[label_key_query]

if train_size is not None:
    # adata_balanced = balance_anndata(adata_train, min_cells_per_class, label_key)
    sc.pp.subsample(adata_train, n_obs=train_size, copy=False, random_state=42)
    # adata_train = ad.concat([adata_train, adata_balanced], axis=0)

accuracy, f1_score_weighted, f1_score_macro, y_pred = evaluate(train_embs, train_labels, test_embs, test_labels, classifier, metric=metric, pred_within_test_labels=False, lr=lr)
print(f'{accuracy:.3f}, {f1_score_weighted:.3f}, {f1_score_macro:.3f}')

removing [] from dataset
Normalization: totalcount_log1p
train split: 15560 cells containing 22 cell types, embeddings of size 128
test split: 13270 cells containing 21 cell types, embeddings of size 128
Fitting KNN (euclidean) with n_neighbors:  10
knn Accuracy: 0.510
knn F1-Weighted: 0.509
knn F1-Macro: 0.377
0.510, 0.509, 0.377


In [29]:
results = []

In [30]:
results.append((config, tissue, dataset_ref, filename_ref, dataset_query, filename_query, classifier, f'{method} {normalization}', f'{method} {normalization}', accuracy, f1_score_weighted, f1_score_macro))

In [31]:
len(results)

1

In [32]:
results_df = pd.concat([results_df, pd.DataFrame(results, columns=columns)])

duplicates = results_df.duplicated(subset=results_df.columns[:-3], keep='last')
print(f"Removing {duplicates.sum()} duplicates")
results_df = results_df[~duplicates]

Removing 0 duplicates


In [33]:
results_df.to_csv(results_path, index=False)

# Celltypist

First select the dataset config and embedding from the previous section

In [78]:
from ct_rep.celltype.celltype_pred import evaluate
from ct_rep.utils.utils import sample_data, adapt_vars
from ct_rep.utils.utils import normalize
import celltypist

In [ ]:
dataset_config = ("neurips_2021_multiome", "adata_gex.h5ad", "batch", "s1", "cell_type", "neurips_2021_multiome", "adata_gex.h5ad", "batch", "s3", "cell_type", "blood")
# dataset_config = ("7840398e-2e12-49d2-b21a-f59a1908057e", "adata_subsample.h5ad", "donor_id", "%", "cell_type", "7840398e-2e12-49d2-b21a-f59a1908057e", "adata_subsample.h5ad", "donor_id", "%", "cell_type", "skeletal muscle")
# dataset_config = ("multiple_sclerosis_scgpt", "c_data.h5ad", "-", "*", "celltype", "multiple_sclerosis_scgpt", "filtered_ms_adata.h5ad", "-", "*", "celltype", "brain (ms lesions)")

config = "_".join([f'[{c}]' for c in dataset_config])
dataset_ref, filename_ref, split_key_ref, split_value_ref, label_key_ref, dataset_query, filename_query, split_key_query, split_value_query, label_key_query, tissue = dataset_config

adata_path_ref = f"/lustre/groups/ml01/projects/contrastive_transformer/data/{dataset_ref}/h5ads/{filename_ref}"
adata_path_query = f"/lustre/groups/ml01/projects/contrastive_transformer/data/{dataset_query}/h5ads/{filename_query}"

In [79]:
method = "CellTypist" # Inner PCA-32, Raw Count Inner
emb_path = "CellTypist" # Inner PCA-32, Raw Count Inner
normalization = "totalcount/log1p" # raw, log1p, totalcount/log1p
classifier = 'linear'
metric = 'euclidean'
min_count = 50
train_size=None

In [80]:
adata_train = sc.read_h5ad(adata_path_ref)
adata_test = sc.read_h5ad(adata_path_query)

# remove low frequency cell types
freq = adata_train.obs[label_key_ref].value_counts(normalize=False)
print(f'removing {list(freq[freq < min_count].index)} from dataset')
adata_train = adata_train[adata_train.obs[label_key_ref].isin(freq[freq > min_count].index)]
adata_test = adata_test[adata_test.obs[label_key_query].isin(freq[freq > min_count].index)]


# split
if split_value_ref=="%":
    assert (adata_train.obs.index == adata_test.obs.index).all(), 'index mismatch'
    train_split, test_split = train_test_split(list(adata_train.obs[split_key_ref].unique()), test_size=0.5, random_state=42)
    print(f'train split: {train_split}, test split: {test_split}')
    adata_train = adata_train[adata_train.obs[split_key_ref].isin(train_split)]
    adata_test = adata_test[adata_test.obs[split_key_query].isin(test_split)]
elif split_value_ref=="*":
    adata_train = adata_train
    adata_test = adata_test
else:
    adata_train = adata_train[adata_train.obs[split_key_ref].str.contains(split_value_ref)]
    adata_test = adata_test[adata_test.obs[split_key_query].str.contains(split_value_query)]


adata_train = adata_train.copy()
adata_test = adata_test.copy()
############################################################################################################
shared_genes = adata_train.var.index.intersection(adata_test.var.index)
adata_train = adata_train[:, shared_genes].copy()
adata_test = adata_test[:, shared_genes].copy()

X_ref = normalize(adata_train.X.toarray(), normalization)
X_query = normalize(adata_test.X.toarray(), normalization)    
############################################################################################################
adata_train.X = X_ref
adata_test.X = X_query

train_labels = adata_train.obs[label_key_ref]
test_labels = adata_test.obs[label_key_query]

if train_size is not None:
    # adata_balanced = balance_anndata(adata_train, min_cells_per_class, label_key)
    sc.pp.subsample(adata_train, n_obs=train_size, copy=False, random_state=42)
    # adata_train = ad.concat([adata_train, adata_balanced], axis=0)

removing [] from dataset


In [ ]:
ct_model = celltypist.train(
    adata_train,
    labels=train_labels, 
    n_jobs=10, 
    feature_selection=True,
    use_SGD=True, 
    use_GPU=True,
    mini_batch=True,
    # batch_number=256,
    with_mean=False,
    random_state=1,
)

🍳 Preparing data before training
🔬 Input data has 15560 cells and 13431 genes
⚖️ Scaling input data
🏋️ Training data using mini-batch SGD logistic regression
⏳ Epochs: [1/10]
⏳ Epochs: [2/10]
⏳ Epochs: [3/10]
⏳ Epochs: [4/10]
⏳ Epochs: [5/10]
⏳ Epochs: [6/10]
⏳ Epochs: [7/10]
⏳ Epochs: [8/10]
⏳ Epochs: [9/10]
⏳ Epochs: [10/10]
🔎 Selecting features
🧬 4763 features are selected
🏋️ Starting the second round of training
🏋️ Training data using mini-batch SGD logistic regression
⏳ Epochs: [1/10]
⏳ Epochs: [2/10]
⏳ Epochs: [3/10]
⏳ Epochs: [4/10]
⏳ Epochs: [5/10]
⏳ Epochs: [6/10]
⏳ Epochs: [7/10]
⏳ Epochs: [8/10]
⏳ Epochs: [9/10]
⏳ Epochs: [10/10]
✅ Model training done!


In [83]:
preds = celltypist.annotate(adata_test, model=ct_model) # majority_voting = True

🔬 Input data has 13270 cells and 13431 genes
🔗 Matching reference genes in the model
🧬 4763 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!


In [84]:
y_pred = preds.predicted_labels.predicted_labels

In [ ]:
accuracy = accuracy_score(test_labels, y_pred)
f1_score_weighted = f1_score(test_labels, y_pred, labels=np.unique(test_labels), average='weighted')
f1_score_macro = f1_score(test_labels, y_pred, labels=np.unique(test_labels), average='macro')
print(f'{classifier} Accuracy: {accuracy:.3f}')
print(f'{classifier} F1-Weighted: {f1_score_weighted:.3f}')
print(f'{classifier} F1-Macro: {f1_score_macro:.3f}')

linear Accuracy: 0.715
linear F1-Weighted: 0.706
linear F1-Macro: 0.637


In [86]:
results = []
results.append((config, tissue, dataset_ref, filename_ref, dataset_query, filename_query, classifier, method, method, accuracy, f1_score_weighted, f1_score_macro))

In [87]:
len(results)

1

In [ ]:
results_df = pd.concat([results_df, pd.DataFrame(results, columns=columns)])

duplicates = results_df.duplicated(subset=results_df.columns[:-3], keep='last')
print(f"Removing {duplicates.sum()} duplicates")
results_df = results_df[~duplicates]

Removing 1 duplicates


In [69]:
results_df.to_csv(results_path, index=False)